In [1]:
import pandas as pd 
import json
import mne
import numpy as np
import mne
import os
from itertools import product
import glob

# --------------------------------------------------------------------------
# REPRODUCIBILITY & HARDWARE SETUP (Must be first)
# ---------------------------------------------------------------------------
print("it started")
import os
import random
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

import numpy as np
import tensorflow as tf
import torch

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.config.threading.set_inter_op_parallelism_threads(1)
tf.config.threading.set_intra_op_parallelism_threads(1)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

# ---------------------------------------------------------------------------
# ORIGINAL IMPORTS & SETUP
# ---------------------------------------------------------------------------
import json
import uuid
import pandas as pd
import matplotlib.pyplot as plt

# Scipy & MNE
import mne
from scipy.signal import stft, welch
from scipy.stats import entropy, norm
from sklearn.model_selection import KFold, train_test_split

# Scikit-learn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.utils.class_weight import compute_class_weight

# TensorFlow / Keras
from tensorflow.keras import layers, models, Model, callbacks

print(f"Reproducibility settings locked with SEED: {SEED}")

# GPU Check
if tf.config.list_physical_devices('GPU'):
    print("TensorFlow GPU Accelerated Backend Active.")
else:
    print("No GPU detected for TensorFlow. Using CPU.")

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA GPU Accelerated Backend Active: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")

it started
Reproducibility settings locked with SEED: 42
No GPU detected for TensorFlow. Using CPU.


2026-08-28 07:47:57.498292: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [2]:
tsv_path = "/kaggle/input/datasets/adithyarajnarayanan/pd-eeg-dataset-2/PD EEG dataset/participants.tsv"

df = pd.read_csv(tsv_path, sep="\t")
print(df.head())

  participant_id GROUP    ID     EEG  AGE GENDER  MOCA  UPDRS  TYPE
0        sub-001    PD  1001  PD1001   80      M    19   28.0     1
1        sub-002    PD  1011  PD1011   81      M    17   25.0     1
2        sub-003    PD  1021  PD1021   68      F    26   10.0     1
3        sub-004    PD  1031  PD1031   80      M    22   10.0     1
4        sub-005    PD  1041  PD1041   56      M    21   13.0     1


In [3]:
set_file_path = "/kaggle/input/datasets/adithyarajnarayanan/pd-eeg-dataset-2/PD EEG dataset/sub-001/eeg/sub-001_task-Rest_eeg.set"

# Load the raw EEG data using MNE
raw = mne.io.read_raw_eeglab(set_file_path, preload=True)
eeg_signals = raw.get_data()

print("Shape of EEG signals array (C, L):", eeg_signals.shape)

Reading /kaggle/input/datasets/adithyarajnarayanan/pd-eeg-dataset-2/PD EEG dataset/sub-001/eeg/sub-001_task-Rest_eeg.fdt
Reading 0 ... 140829  =      0.000 ...   281.658 secs...
Shape of EEG signals array (C, L): (63, 140830)


/tmp/ipykernel_131/2417030768.py:4: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_file_path, preload=True)


In [4]:
channel_names = raw.ch_names
print(f"Total number of channels: {len(channel_names)}")
print("Channel names:")
print(", ".join([f"{i+1}: {ch}" for i, ch in enumerate(channel_names)]))

Total number of channels: 63
Channel names:
1: Fp1, 2: Fz, 3: F3, 4: F7, 5: FT9, 6: FC5, 7: FC1, 8: C3, 9: T7, 10: TP9, 11: CP5, 12: CP1, 13: P3, 14: P7, 15: O1, 16: Oz, 17: O2, 18: P4, 19: P8, 20: TP10, 21: CP6, 22: CP2, 23: Cz, 24: C4, 25: T8, 26: FT10, 27: FC6, 28: FC2, 29: F4, 30: F8, 31: Fp2, 32: AF7, 33: AF3, 34: AFz, 35: F1, 36: F5, 37: FT7, 38: FC3, 39: C1, 40: C5, 41: TP7, 42: CP3, 43: P1, 44: P5, 45: PO7, 46: PO3, 47: POz, 48: PO4, 49: PO8, 50: P6, 51: P2, 52: CPz, 53: CP4, 54: TP8, 55: C6, 56: C2, 57: FC4, 58: FT8, 59: F6, 60: AF8, 61: AF4, 62: F2, 63: FCz


In [5]:
occipital_channels = [str(ch) for ch in ['O1', 'O2', 'Oz', 'PO7', 'PO8', 'POz'] if ch in ['AF3', 'AF4', 'AF7', 'AF8', 'AFz', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'CP1', 'CP2', 'CP3', 'CP4', 'CP5', 'CP6', 'CPz', 'Cz', 'F1', 'F2', 'F3', 'F4', 'F5', 'F6', 'F7', 'F8', 'FC1', 'FC2', 'FC3', 'FC4', 'FC5', 'FC6', 'FCz', 'FT10', 'FT7', 'FT8', 'Fp1', 'Fp2', 'Fz', 'O1', 'O2', 'Oz', 'P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'PO7', 'PO8', 'POz', 'T7', 'T8', 'TP10', 'TP7', 'TP8', 'TP9']]
raw.pick(occipital_channels)

<RawEEGLAB | sub-001_task-Rest_eeg.fdt, 6 x 140830 (281.7 s), ~6.5 MiB, data loaded>

In [6]:
def load_segment_set(set_file_path,l_freq,h_freq,target_sfreq=256, window_sec=2, overlap_ratio=0.5, peak_to_peak_threshold=0.00028):
    
    # Load recording (using read_raw_eeglab for .set/.fdt files)
    raw = mne.io.read_raw_eeglab(set_file_path, preload=True, verbose=False)

    # Define the precise 63 channel order requested
    target_channels = occipital_channels

    # Reorder and pick the specific channels
    raw.pick(target_channels)

    # 2 & 3. Bandpass Filter (0.5 to 45 Hz)
    raw.filter(l_freq=0.5, h_freq=45.0, fir_design='firwin', verbose=False)

    # 4. Notch Filter at 50 Hz to eliminate line noise
    raw.notch_filter(freqs=50.0, fir_design='firwin', verbose=False)

    # 5. Common Average Reference (CAR)
    raw.set_eeg_reference(ref_channels='average', verbose=False)

    # 6. Resample to target frequency
    raw.resample(target_sfreq, verbose=False)

    # Get data matrix
    signals = raw.get_data()
    print("Full signal shape (C, L):", signals.shape)
    C, L = signals.shape
    window_samples = int(window_sec * target_sfreq)

    # Calculate stride samples based on the overlap ratio (e.g., 0.5 means 50% overlap)
    stride_samples = int(window_samples * (1 - overlap_ratio))
    if stride_samples < 1:
        stride_samples = 1

    # 7. Generate sequential windows & Apply Artifact Rejection
    window_list = []
    start = 0
    while start + window_samples <= L:
        end = start + window_samples
        window = signals[:, start:end]
        
        # 8. Peak-to-Peak Threshold Artifact Rejection
        peak_to_peak = np.ptp(window, axis=1)
        if np.any(peak_to_peak > peak_to_peak_threshold):
            start += stride_samples
            continue
        
        window_list.append(window)
        start += stride_samples

    # Check if any windows were created
    if len(window_list) == 0:
        return np.empty((0, C, window_samples))

    # Convert to standard array format (N, C, T)
    windows = np.array(window_list)
    if l_freq >0 and h_freq>0:
        windows = mne.filter.filter_data(data=windows, sfreq=256, l_freq=l_freq, h_freq=h_freq, method='iir',verbose=False)

    return windows

In [7]:
def get_data(l_freq, h_freq, peak_to_peak_threshold=0.00028):
    X_pd = []
    X_hc = []
    
    # Base directory path for the dataset
    base_dir = "/kaggle/input/datasets/adithyarajnarayanan/pd-eeg-dataset-2/PD EEG dataset"
    
    # Loop through subject indices from 1 to 149
    for sub_id in range(1, 150):
        print("patient number is", sub_id)
        sub_str = f"sub-{sub_id:03d}"
        set_file_path = os.path.join(base_dir, sub_str, "eeg", f"{sub_str}_task-Rest_eeg.set")
        
        # Check if file exists before attempting to load
        if not os.path.exists(set_file_path):
            print(f"File not found for subject {sub_id}")
            continue
            
        if sub_id < 101:
            windows = load_segment_set(
                set_file_path=set_file_path, 
                l_freq=l_freq, 
                h_freq=h_freq, 
                target_sfreq=256, 
                window_sec=2, 
                overlap_ratio=0.0,
                peak_to_peak_threshold=peak_to_peak_threshold
            )
            print(windows.shape)
            if windows.shape[0] > 0:
                X_pd.append(windows)
            else:
                print("no enough windows subject number", sub_id)
                
        else:
            windows = load_segment_set(
                set_file_path=set_file_path, 
                l_freq=l_freq, 
                h_freq=h_freq, 
                target_sfreq=256, 
                window_sec=2, 
                overlap_ratio=0,
                peak_to_peak_threshold=peak_to_peak_threshold
            )
            print(windows.shape)
            if windows.shape[0] > 0:
                X_hc.append(windows)
            else:
                print("no enough windows subject number", sub_id)
                
    return X_hc, X_pd

In [8]:
def balance_matrices_subject_wise(X_list_c0, X_list_c1):
    c0_windows_per_sub = [sub.shape[0] for sub in X_list_c0]
    c1_windows_per_sub = [sub.shape[0] for sub in X_list_c1]
    
    total_c0 = sum(c0_windows_per_sub)
    total_c1 = sum(c1_windows_per_sub)
    
    if total_c0 == total_c1:
        return np.concatenate(X_list_c0, axis=0), np.concatenate(X_list_c1, axis=0)

    if total_c1 > total_c0:
        maj_list = X_list_c1
        maj_counts = np.array(c1_windows_per_sub)
        target_total = total_c0
        is_c1_majority = True
    else:
        maj_list = X_list_c0
        maj_counts = np.array(c0_windows_per_sub)
        target_total = total_c1
        is_c1_majority = False

    num_maj_subs = len(maj_list)
    allocations = np.zeros(num_maj_subs, dtype=int)
    remaining_target = target_total
    active_subs = np.ones(num_maj_subs, dtype=bool)

    while remaining_target > 0 and np.any(active_subs):
        num_active = np.sum(active_subs)
        base_share = remaining_target // num_active
        remainder = remaining_target % num_active
        
        if base_share == 0:
            chosen_indices = np.where(active_subs)[0][:remaining_target]
            for idx in chosen_indices:
                allocations[idx] += 1
            break
            
        for i in range(num_maj_subs):
            if active_subs[i]:
                share = base_share + (1 if remainder > 0 else 0)
                remainder -= 1 if remainder > 0 else 0
                
                available = maj_counts[i] - allocations[i]
                take = min(share, available)
                
                allocations[i] += take
                remaining_target -= take
                
                if allocations[i] == maj_counts[i]:
                    active_subs[i] = False

    processed_maj_list = []
    rng = np.random.default_rng(SEED)
    for i, sub_windows in enumerate(maj_list):
        n_needed = allocations[i]
        if n_needed > 0:
            chosen_indices = rng.choice(sub_windows.shape[0], size=n_needed, replace=False)
            processed_maj_list.append(sub_windows[chosen_indices])
            
    X_processed_maj = np.concatenate(processed_maj_list, axis=0)

    if is_c1_majority:
        return np.concatenate(X_list_c0, axis=0), X_processed_maj
    else:
        return X_processed_maj, np.concatenate(X_list_c1, axis=0)

In [9]:
def scale_data(X_list):
    scaled = []
    for sub in X_list:
        flat = sub.reshape(-1, sub.shape[-1])
        mu = np.mean(flat, axis=0)
        std = np.std(flat, axis=0) + 1e-8
        scaled.append((sub - mu) / std)
    return scaled

In [10]:


class PureSpectralConv1D(layers.Layer):
    """
    Pure 1D Fourier Integral Operator:
    Computes (K(a) * v)(t) = F^-1( R_phi * F(v) )(t) continuously across domain resolution.
    """
    def __init__(self, in_channels, out_channels, modes1, **kwargs):
        super(PureSpectralConv1D, self).__init__(**kwargs)
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1
        self.scale = 1.0 / (in_channels * out_channels)

    def build(self, input_shape):
        self.weights1_real = self.add_weight(
            shape=(self.in_channels, self.out_channels, self.modes1),
            initializer=tf.random_normal_initializer(stddev=self.scale),
            trainable=True, name="w_real"
        )
        self.weights1_imag = self.add_weight(
            shape=(self.in_channels, self.out_channels, self.modes1),
            initializer=tf.random_normal_initializer(stddev=self.scale),
            trainable=True, name="w_imag"
        )
        super(PureSpectralConv1D, self).build(input_shape)

    def call(self, x):
        time_steps = tf.shape(x)[1]
        
        # 1. Continuous Fourier Transform
        x_transposed = tf.transpose(x, perm=[0, 2, 1])
        x_ft = tf.signal.rfft(x_transposed)
        
        # 2. Spectral Kernel Multiplication R_phi (Lower modes operator)
        x_ft_sub = x_ft[:, :, :self.modes1]
        weights1 = tf.complex(self.weights1_real, self.weights1_imag)
        out_ft = tf.einsum("bix,iox->box", x_ft_sub, weights1)
        
        # 3. Dynamic Zero-Padding for exact resolution recovery
        pad_len = (time_steps // 2 + 1) - self.modes1
        paddings = tf.stack([
            tf.constant([0, 0]), 
            tf.constant([0, 0]), 
            tf.stack([0, pad_len])
        ])
        out_ft_padded = tf.pad(out_ft, paddings)
        
        # 4. Inverse Fourier Transform back to temporal function space
        x_out = tf.signal.irfft(out_ft_padded, fft_length=[time_steps])
        return tf.transpose(x_out, perm=[0, 2, 1])


class PureFNO1D(Model):
    """
    Pure 1D Fourier Neural Operator (FNO) Architecture:
    Function-to-Function Mapping Framework G: A -> U
    
    Structure:
    1. Lifting Operator (P): Maps channel space to high-dim continuous space
    2. Stacked Fourier Layers (K_i + W_i): Operates strictly in function space
    3. Projection Operator (Q): Maps latent representation to continuous output field u(t)
    4. Domain Integration: Approximates continuous integral int_Omega u(t) dt
    """
    def __init__(self, in_channels=16, modes=12, width=32, **kwargs):
        super(PureFNO1D, self).__init__(**kwargs)
        
        # Permute (N, C, T) -> (N, T, C)
        self.permute = layers.Permute((2, 1))
        
        # 1. LIFTING OPERATOR P: a(t) -> v_0(t)
        self.p_lifting = layers.Dense(width)
        
        # 2. FOURIER OPERATOR LAYER 1: v_0(t) -> v_1(t)
        self.fno1 = PureSpectralConv1D(in_channels=width, out_channels=width, modes1=modes)
        self.w1 = layers.Conv1D(width, kernel_size=1)  # Local spatial linear transformation W
        
        # FOURIER OPERATOR LAYER 2: v_1(t) -> v_2(t)
        self.fno2 = PureSpectralConv1D(in_channels=width, out_channels=width, modes1=modes)
        self.w2 = layers.Conv1D(width, kernel_size=1)
        
        # 3. PROJECTION OPERATOR Q: v_2(t) -> u(t) (Target Continuous Field)
        self.q_proj1 = layers.Dense(128, activation='gelu')
        self.q_proj2 = layers.Dense(1)  # Evaluates output continuous field u(t)
        
    def call(self, inputs):
        # inputs shape: (N, Channels, Time)
        x = self.permute(inputs)  # -> (N, Time, Channels)
        
        # Step 1: Lift to higher-dimensional continuous space
        x = self.p_lifting(x)
        
        # Step 2: Pass through Non-linear Fourier Integral Operators
        # Layer 1: v_1 = activation( K_1(v_0) + W_1(v_0) )
        x = tf.nn.gelu(self.fno1(x) + self.w1(x))
        
        # Layer 2: v_2 = activation( K_2(v_1) + W_2(v_1) )
        x = tf.nn.gelu(self.fno2(x) + self.w2(x))
        
        # Step 3: Project back to target scalar field u(t) across domain
        x = self.q_proj1(x)
        u_t = self.q_proj2(x)  # Shape: (N, Time, 1) - Continuous functional field
        
        # Step 4: Domain Integration (Continuous Operator Functional Evaluation)
        # Numerical approximation of integral over domain: Y = Sigmoid( 1/T * int_0^T u(t) dt )
        domain_integral = tf.reduce_mean(u_t, axis=1)  # Shape: (N, 1)
        return tf.nn.sigmoid(domain_integral)

In [11]:

def run_subject_level_mc_cv_optimized(X_healthy, X_pd, SEED=42):
    X_healthy = scale_data(X_healthy)
    X_pd = scale_data(X_pd)
    
    # Automatically infer channel count from input arrays (shape: N_epochs, Channels, Time)
    n_channels = X_healthy[0].shape[1]
    
    n_hc, n_pd = len(X_healthy), len(X_pd)
    outer_kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    thresholds = list(range(65, 95, 5))
    
    hc_splits = list(outer_kf.split(np.arange(n_hc)))
    pd_splits = list(outer_kf.split(np.arange(n_pd)))
    
    total_correct = 0
    total_subjects = 0
    fold_summary_records = []
    
    # Pure FNO Hyperparameter Grid Search (Function-to-Function Operator Space)
    param_grid = {
        'lr': [1e-3],
        'batch_size': [32],
        'modes': [12],
        'width': [32]
    }
    
    keys = param_grid.keys()
    all_combinations = [dict(zip(keys, combo)) for combo in product(*param_grid.values())]
    
    for fold in range(5):
        print(f"\n========================================")
        print(f"========== OUTER FOLD {fold+1} / 5 ==========")
        print(f"========================================")
        
        hc_train_all, hc_test = hc_splits[fold]
        pd_train_all, pd_test = pd_splits[fold]
        
        best_score = -1.0
        best_params = None
        best_threshold = 75
        
        hc_inner_splits = list(KFold(n_splits=3, shuffle=True, random_state=SEED).split(hc_train_all))
        pd_inner_splits = list(KFold(n_splits=3, shuffle=True, random_state=SEED).split(pd_train_all))
        
        for params in all_combinations:
            inner_fold_accuracies = []
            inner_fold_thresholds = []
            
            for inner_fold in range(3):
                hc_tr_in_idx, hc_val_in_idx = hc_inner_splits[inner_fold]
                pd_tr_in_idx, pd_val_in_idx = pd_inner_splits[inner_fold]
                
                hc_train_sub = [X_healthy[hc_train_all[i]] for i in hc_tr_in_idx]
                pd_train_sub = [X_pd[pd_train_all[i]] for i in pd_tr_in_idx]
                hc_val_sub = [X_healthy[hc_train_all[i]] for i in hc_val_in_idx]
                pd_val_sub = [X_pd[pd_train_all[i]] for i in pd_val_in_idx]
                
                # Balance classes for inner training
                X_tr_hc_bal, X_tr_pd_bal = balance_matrices_subject_wise(hc_train_sub, pd_train_sub)
                X_inner_train = np.concatenate([X_tr_hc_bal, X_tr_pd_bal], axis=0)
                y_inner_train = np.concatenate([np.zeros(len(X_tr_hc_bal)), np.ones(len(X_tr_pd_bal))], axis=0)
                
                # Shuffle training data
                shuffle_idx = np.random.RandomState(SEED).permutation(len(X_inner_train))
                X_inner_train = X_inner_train[shuffle_idx]
                y_inner_train = y_inner_train[shuffle_idx]
                
                # Explicit Train/Val split
                val_size = int(len(X_inner_train) * 0.1)
                X_tr, y_tr = X_inner_train[val_size:], y_inner_train[val_size:]
                X_va, y_va = X_inner_train[:val_size], y_inner_train[:val_size]

                # Instantiate Pure FNO Model (Continuous Operator Mapping)
                inner_model = PureFNO1D(
                    in_channels=n_channels,
                    modes=params['modes'], 
                    width=params['width']
                )
                inner_model.compile(
                    optimizer=tf.keras.optimizers.Adam(learning_rate=params['lr']), 
                    loss='binary_crossentropy',
                    metrics=['accuracy']
                )
                
                early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
                inner_model.fit(
                    X_tr, y_tr, 
                    epochs=40, batch_size=params['batch_size'], 
                    verbose=0, validation_data=(X_va, y_va), callbacks=[early_stop]
                )
                
                # Inner validation threshold tuning
                val_subjects = hc_val_sub + pd_val_sub
                val_labels = [0]*len(hc_val_sub) + [1]*len(pd_val_sub)
                
                val_subject_ratios = []
                valid_val_labels = []
                
                for sub, true_lbl in zip(val_subjects, val_labels):
                    sub_array = np.asarray(sub, dtype=np.float32)
                    
                    if sub_array.ndim == 2:
                        sub_array = np.expand_dims(sub_array, axis=0)
                        
                    if sub_array.shape[0] == 0:
                        continue
                        
                    epoch_probs = inner_model.predict(sub_array, batch_size=params['batch_size'], verbose=0).flatten()
                    pct_pd = float(np.mean(epoch_probs) * 100)
                    val_subject_ratios.append(pct_pd)
                    valid_val_labels.append(true_lbl)

                best_t_inner, max_inner_acc = 75, -1.0
                for t in thresholds:
                    t_preds = [1 if ratio >= t else 0 for ratio in val_subject_ratios]
                    acc = accuracy_score(valid_val_labels, t_preds) if len(valid_val_labels) > 0 else 0.0
                    if acc > max_inner_acc:
                        max_inner_acc = acc
                        best_t_inner = t
                
                inner_fold_accuracies.append(max_inner_acc)
                inner_fold_thresholds.append(best_t_inner)
            
            mean_inner_acc = np.mean(inner_fold_accuracies)
            if mean_inner_acc > best_score:
                best_score = mean_inner_acc
                best_params = params
                # Take median of inner thresholds to preserve steps of 5
                best_threshold = int(np.median(inner_fold_thresholds))
        
        print(f">> Best Grid Parameters Selected: {best_params} | Threshold: {best_threshold}% (Inner Acc: {best_score:.4f})")
        
        # --- OUTER TRAINING & TESTING ---
        hc_train_final = [X_healthy[i] for i in hc_train_all]
        pd_train_final = [X_pd[i] for i in pd_train_all]
        
        X_tr_hc_final, X_tr_pd_final = balance_matrices_subject_wise(hc_train_final, pd_train_final)
        X_train_final = np.concatenate([X_tr_hc_final, X_tr_pd_final], axis=0)
        y_train_final = np.concatenate([np.zeros(len(X_tr_hc_final)), np.ones(len(X_tr_pd_final))], axis=0)
        
        shuffle_idx_final = np.random.RandomState(SEED).permutation(len(X_train_final))
        X_train_final = X_train_final[shuffle_idx_final]
        y_train_final = y_train_final[shuffle_idx_final]
        
        val_size_final = int(len(X_train_final) * 0.1)
        X_tr_f, y_tr_f = X_train_final[val_size_final:], y_train_final[val_size_final:]
        X_va_f, y_va_f = X_train_final[:val_size_final], y_train_final[:val_size_final]

        final_model = PureFNO1D(
            in_channels=n_channels,
            modes=best_params['modes'], 
            width=best_params['width']
        )
        final_model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=best_params['lr']), 
            loss='binary_crossentropy',
            metrics=['accuracy']
        )
        
        early_stop_final = callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
        final_model.fit(
            X_tr_f, y_tr_f, 
            epochs=80, batch_size=best_params['batch_size'], 
            verbose=0, validation_data=(X_va_f, y_va_f), callbacks=[early_stop_final]
        )
        
        test_subjects = [X_healthy[i] for i in hc_test] + [X_pd[i] for i in pd_test]
        test_labels = [0]*len(hc_test) + [1]*len(pd_test)
        n_hc_test = len(hc_test)
        n_pd_test = len(pd_test)
        
        hc_correct_count = 0
        pd_correct_count = 0
        
        for sub, true_label in zip(test_subjects, test_labels):
            sub_array = np.asarray(sub, dtype=np.float32)
            if sub_array.ndim == 2:
                sub_array = np.expand_dims(sub_array, axis=0)
                
            if sub_array.shape[0] == 0:
                continue
                
            pct_pd = float(np.mean(final_model.predict(sub_array, batch_size=best_params['batch_size'], verbose=0).flatten()) * 100)
            
            vote_thresholds = [best_threshold - 5, best_threshold, best_threshold + 5]
            votes = [1 if pct_pd >= t else 0 for t in vote_thresholds]
            pred = 1 if sum(votes) >= 2 else 0
            
            if pred == true_label:
                if true_label == 0:
                    hc_correct_count += 1
                else:
                    pd_correct_count += 1
                    
        fold_total_correct = hc_correct_count + pd_correct_count
        fold_total_subjects = len(test_subjects)
        
        total_correct += fold_total_correct
        total_subjects += fold_total_subjects
        
        fold_summary_records.append({
            'Fold Number': fold + 1,
            'Optimal Hyperparams': str(best_params),
            'Healthy Correct': f"{hc_correct_count}/{n_hc_test}",
            'PD Correct': f"{pd_correct_count}/{n_pd_test}",
            'Total Correct': f"{fold_total_correct}/{fold_total_subjects}"
        })
        
        print(f"Outer Fold {fold+1} Stats -> Healthy: {hc_correct_count}/{n_hc_test} | PD: {pd_correct_count}/{n_pd_test} | Total: {fold_total_correct}/{fold_total_subjects}")

    summary_df = pd.DataFrame(fold_summary_records)
    print(f"\n========================================")
    print(f"Total Combined Correct: {total_correct}/{total_subjects}")
    print("\n--- Nested Cross-Validation Summary ---")
    print(summary_df.to_string(index=False))
    
    return summary_df

In [12]:
X_hc,X_pd = get_data(8,12)
df = run_subject_level_mc_cv_optimized(X_hc, X_pd, SEED=42)
print(df)

patient number is 1


/tmp/ipykernel_131/803186098.py:4: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_file_path, preload=True, verbose=False)


Full signal shape (C, L): (6, 72105)
(139, 6, 512)
patient number is 2


/tmp/ipykernel_131/803186098.py:4: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(set_file_path, preload=True, verbose=False)


Full signal shape (C, L): (6, 83466)
(163, 6, 512)
patient number is 3


...

Outer Fold 4 Stats -> Healthy: 8/10 | PD: 3/20 | Total: 11/30

========== OUTER FOLD 5 / 5 ==========


2026-08-28 09:18:33.920407: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-08-28 09:22:37.750581: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

>> Best Grid Parameters Selected: {'lr': 0.001, 'batch_size': 32, 'modes': 12, 'width': 32} | Threshold: 65% (Inner Acc: 0.5090)


2026-08-28 09:32:15.633015: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-08-28 09:39:28.641982: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

Outer Fold 5 Stats -> Healthy: 7/9 | PD: 8/20 | Total: 15/29

Total Combined Correct: 71/149

--- Nested Cross-Validation Summary ---
 Fold Number                                       Optimal Hyperparams Healthy Correct PD Correct Total Correct
           1 {'lr': 0.001, 'batch_size': 32, 'modes': 12, 'width': 32}            5/10       9/20         14/30
           2 {'lr': 0.001, 'batch_size': 32, 'modes': 12, 'width': 32}            6/10      14/20         20/30
           3 {'lr': 0.001, 'batch_size': 32, 'modes': 12, 'width': 32}            8/10       3/20         11/30
           4 {'lr': 0.001, 'batch_size': 32, 'modes': 12, 'width': 32}            8/10       3/20         11/30
           5 {'lr': 0.001, 'batch_size': 32, 'modes': 12, 'width': 32}             7/9       8/20         15/29
   Fold Number                                Optimal Hyperparams  \
0            1  {'lr': 0.001, 'batch_size': 32, 'modes': 12, '...   
1            2  {'lr': 0.001, 'batch_size': 32, 'modes':